In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

#importlib.reload(experimenter)

In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [9]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [10]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

[std] Building: 3/3 (100%) ✓ Complete


In [11]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

        annual_income  debt_to_income_ratio  credit_score   loan_amount  \
id                                                                        
395367   71433.867188                 0.219           639  17829.300781   
531644   25336.169922                 0.063           674   7497.600098   
91834    50857.179688                 0.192           754  12072.650391   
115491   51449.031250                 0.092           575  24997.080078   
484339    8589.000000                 0.162           699  12359.660156   
...               ...                   ...           ...           ...   
329121   43466.250000                 0.216           572   6605.750000   
425024   18079.039062                 0.058           797   5102.259766   
405150   35997.011719                 0.060           689  12197.589844   
571258   24325.119141                 0.148           675  12122.629883   
259356   68570.257812                 0.099           733  13370.679688   

        interest_rate  g

In [12]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [13]:
for z in e.get_data(0,  [('std', None)]):
    print(z)

((<modeler._data_wrapper.PandasWrapper object at 0x70bc0af373e0>, <modeler._data_wrapper.PandasWrapper object at 0x70bc0af37b90>), <modeler._data_wrapper.PandasWrapper object at 0x70bc0af379b0>)


In [14]:
e.set_node('lr1', 'lr')

[lr1] Building: 3/3 (100%) ✓ Complete


In [15]:
for (train_, train_v_), valid_ in e.get_data(0, e.nodes['std'].edges):
    print(train_, train_v_, valid_)

<modeler._data_wrapper.PandasWrapper object at 0x70bc109c6630> <modeler._data_wrapper.PandasWrapper object at 0x70bc0af36090> <modeler._data_wrapper.PandasWrapper object at 0x70bc0af354c0>


In [16]:
e.set_node('lr1', 'lr')

⚠️  Updating existing node 'lr1'
[lr1] Building: 3/3 (100%) ✓ Complete


In [17]:
e.get_data(1, edges= [('lr1', None)])

<generator object Experimenter.get_data.<locals>.ret_data_func at 0x70bc0af58160>

In [18]:
from sklearn.preprocessing import StandardScaler
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})

[ohe] Building: 3/3 (100%) ✓ Complete


In [19]:
from modeler import col

In [20]:
e.set_node('lr2', 'lr', edges = [('ohe', col.ohe_drop_first)])

[lr2] Building: 3/3 (100%) ✓ Complete


In [21]:
e.get_data(1, edges= [('lr2', None)])

<generator object Experimenter.get_data.<locals>.ret_data_func at 0x70bc0af584c0>

In [22]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [23]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [24]:
e.add_grp('dim_reduction', method = 'transform')

In [25]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

[pca] Building: 3/3 (100%) ✓ Complete


In [26]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_dim_reduction["dim_reduction"]
        node_pca["pca"]
        style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
    grp_preprocessor --> grp_dim_reduction
```

In [27]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

[lr3] Building: 3/3 (100%) ✓ Complete


In [28]:
e.nodes['pca'].objs_

[[(<modeler._node_processor.TransformProcessor at 0x70bc0af8d4f0>,
   None,
   {'build_id': '4dce3a36-4e2a-43a7-adb7-e038a2adceb0',
    'fit_time': 0.0014600753784179688,
    'train_shape': (3564, 6),
    'train_v_shape': (396, 6)})],
 [(<modeler._node_processor.TransformProcessor at 0x70bc0af51640>,
   None,
   {'build_id': '632880b6-a649-4363-afa7-b206b0a52789',
    'fit_time': 0.002393484115600586,
    'train_shape': (3564, 6),
    'train_v_shape': (396, 6)})],
 [(<modeler._node_processor.TransformProcessor at 0x70bc0af50b60>,
   None,
   {'build_id': 'ec65de1d-419a-42e3-b63a-e32605bd636f',
    'fit_time': 0.0011339187622070312,
    'train_shape': (3564, 6),
    'train_v_shape': (396, 6)})]]

In [29]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (4 path(s) found)**

In [30]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (4 path(s) found)**

In [31]:
e.nodes['std'].objs_[0][0][0].X_

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'grade_subgrade_no']

In [32]:
e.nodes['std'].objs_[0][0][0].output_vars

['std__annual_income',
 'std__debt_to_income_ratio',
 'std__credit_score',
 'std__loan_amount',
 'std__interest_rate',
 'std__grade_subgrade_no']

In [33]:
e.nodes['lr2'].objs_[0][0][0].output_vars

['lr2__loan_paid_back_0', 'lr2__loan_paid_back_1']

In [34]:
e.nodes['lr2'].objs_[0][0][0].obj.classes_

array([0, 1], dtype=int8)

In [35]:
e.nodes['lr3']._fit()

[lr3] Building: 3/3 (100%) ✓ Complete


In [36]:
e.set_node('lr1', grp = 'lr', edges = [('std', None)])

⚠️  Updating existing node 'lr1'
[lr1] Building: 3/3 (100%) ✓ Complete


In [37]:
e.set_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [])

✅ Cycle check passed for all nodes in group 'lr'
🔄 Rebuilding 3 node(s) affected by group 'lr' update
  ├─ Rebuilding 'lr1' (priority: 1)...
⚠️  Updating existing node 'lr1'
[lr1] Building: 3/3 (100%) ✓ Complete
  ├─ Rebuilding 'lr2' (priority: 1)...
⚠️  Updating existing node 'lr2'
[lr2] Building: 3/3 (100%) ✓ Complete
  ├─ Rebuilding 'lr3' (priority: 1)...
⚠️  Updating existing node 'lr3'
[lr3] Building: 3/3 (100%) ✓ Complete
✅ Rebuild complete!


In [38]:
e.nodes['lr1'].org_attr

{'processor': None,
 'edges': [('std', None)],
 'X': None,
 'y': None,
 'method': None,
 'adapter': 'default',
 'params': {}}

In [39]:
e.grps['lr'].get_attrs()

{'edges': [(None, ['loan_paid_back'])],
 'processor': sklearn.linear_model._logistic.LogisticRegression,
 'X': None,
 'y': 'loan_paid_back',
 'method': 'predict_proba',
 'params': {}}

In [40]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [41]:
e.add_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [42]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params={'cat_features': X_cat})

[cb1] Building: 3/3 (100%) ✓ Complete


In [43]:
import lightgbm as lgb

In [44]:
e.add_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [45]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

  Progress: 40/100 (40.0%) | training-binary_logloss: 0.1509, valid_1-binary_logloss: 0.244259

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridde

[lgb1] Building: 3/3 (100%) ✓ Completeg-binary_logloss: 0.0738, valid_1-binary_logloss: 0.2509


In [46]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_.keys()

dict_keys(['learn', 'validation_0', 'validation_1'])

In [47]:
e.desc_node_vars('cb1', 0)

[(['annual_income',
   'debt_to_income_ratio',
   'credit_score',
   'loan_amount',
   'interest_rate',
   'grade_subgrade_no',
   'gender',
   'marital_status',
   'education_level',
   'employment_status',
   'loan_purpose'],
  ['cb1__loan_paid_back_0', 'cb1__loan_paid_back_1'],
  [0])]

In [47]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_['validation_1']

{'Logloss': [0.645759441246397,
  0.6130065322875022,
  0.5725020987063175,
  0.5380210558266334,
  0.5149434214939483,
  0.48901834322816917,
  0.4658994928719027,
  0.44223958047939793,
  0.42341022944119033,
  0.40600890202352247,
  0.3896425477733485,
  0.3774130169578602,
  0.3663380082866546,
  0.3559981144487788,
  0.34591239985819444,
  0.33750671270033883,
  0.3304182506563109,
  0.3235921489014019,
  0.316973691886976,
  0.3126578450679723,
  0.3086949796067106,
  0.30496437044762953,
  0.30027004431766324,
  0.29787817742148065,
  0.2956063172061011,
  0.2936174437478543,
  0.29141830382467176,
  0.28936286889222007,
  0.28646100287579246,
  0.28382234089728553,
  0.2814016968297356,
  0.2787614831277691,
  0.2777716419933787,
  0.27603540913433566,
  0.2748312398649252,
  0.2743936373089017,
  0.2726353836552194,
  0.2711474006610633,
  0.2701325128905084,
  0.2686668357820882,
  0.26772497398803313,
  0.26714451524166843,
  0.2665280284599556,
  0.26592385103015403,
  0.26

In [48]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_cb1["cb1"]
        cb1_dummy[ ]
        style cb1_dummy fill:none,stroke:none
    end
    style node_cb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_cb1
```

**Path from Root to 'cb1' (1 path(s) found)**

In [49]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_dim_reduction["dim_reduction"]
        node_pca["pca"]
        style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_dim_reduction --> grp_clf
    grp_preprocessor --> grp_clf
    grp_preprocessor --> grp_dim_reduction
```

In [50]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lgb1["lgb1"]
        lgb1_dummy[ ]
        style lgb1_dummy fill:none,stroke:none
    end
    style node_lgb1 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    Root --> node_lgb1
```

**Path from Root to 'lgb1' (1 path(s) found)**

In [51]:
e._find_descendants('std')

{'lr1', 'lr3', 'pca'}

In [52]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [53]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_dim_reduction["dim_reduction"]
        node_pca["pca"]
        style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["3 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_cb["cb"]
            grp_cb_count["1 node(s)"]
            style grp_cb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_cb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
        subgraph grp_lgb["lgb"]
            grp_lgb_count["1 node(s)"]
            style grp_lgb_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lgb fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_dim_reduction --> grp_clf
    grp_preprocessor --> grp_clf
    grp_preprocessor --> grp_dim_reduction
```

In [54]:
from modeler import create_like

In [55]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...
   ├─ Created base Experimenter with 1 fold(s)
   ├─ Cloned 6 group(s)
[std] Building: 2/2 (100%) ✓ Complete
[lr1] Building: 2/2 (100%) ✓ Complete
[ohe] Building: 2/2 (100%) ✓ Complete
[lr2] Building: 2/2 (100%) ✓ Complete
[pca] Building: 2/2 (100%) ✓ Complete
[lr3] Building: 2/2 (100%) ✓ Complete
[cb1] Building: 2/2 (100%) ✓ Complete
[lgb1] Building: 2/2 (100%) ✓ Completeg-binary_logloss: 0.0426, valid_1-binary_logloss: 0.3078
   └─ Cloned 8 node(s)
✅ Structure cloning complete!


/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [56]:
for t, v in e.get_data_train(0, [('lr1', slice(-1, None)), ('cb1', slice(-1, None)), (None, [y])]):
    print(t.data, v)

        lr1__loan_paid_back_1  cb1__loan_paid_back_1  loan_paid_back
id                                                                  
395367               0.513990               0.407739               0
531644               0.902688               0.968680               1
91834                0.857890               0.036189               0
115491               0.663619               0.909650               1
484339               0.820404               0.982151               1
...                       ...                    ...             ...
329121               0.372073               0.601110               0
425024               0.980163               0.991319               1
405150               0.914010               0.977182               1
571258               0.794893               0.902436               1
259356               0.926022               0.977036               1

[3564 rows x 3 columns] <modeler._data_wrapper.PandasWrapper object at 0x70bc03345e20>


In [57]:
for (true_train, true_train_v), (prd_train, prd_train_v) in  zip(
    e.get_node_train_output(0, None, [y]),
    e.get_node_train_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_train.data, prd_train.data), 
        roc_auc_score(true_train_v.data, prd_train_v.data)
    )

0.9411658860340462 0.9119514435171505


In [58]:
for true_valid, prd_valid in  zip(
    e.get_node_valid_output(0, None, [y]),
    e.get_node_valid_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_valid.data, prd_valid.data)
    )

0.9253651719043245


In [59]:
class Metric:
    def __init__(
        self, e, target_edge, output_var, metric_func, include_train = False
    ):
        self.e = e
        self.target_edge = target_edge
        self.output_var = output_var
        self.include_train = include_train
        self.metric_func = metric_func
        self.result = {}
        self.build_ids = {}

    def calc_idx(self, nodes, idx):
        result = {}
        grps = {}
        if self.include_train:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and (node, idx) in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_output(idx, None, [y]), self.e.get_node_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for no, ((true_train, true_valid), (prd_train, prd_valid)) in enumerate(iterator):
                    result_train = self.metric_func(true_train[0].data, prd_train[0].data)
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)
                    if true_train[1] is not None:
                        result_sub = {
                            (idx, 'train', f'train_{no}'): result_train,
                            (idx, 'train', f'valid_{no}'): self.metric_func(true_train[1].data, prd_train[1].data),
                            (idx, 'valid', ''): result_valid
                        }
                    else:
                        result_sub = {
                            (idx, 'train'): result_train, (idx, 'valid'): result_valid
                        }
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)
        else:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and node in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_valid_output(idx, None, [y]), self.e.get_node_valid_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for true_valid, prd_valid in iterator:
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)                    
                    result_sub = {idx: result_valid}
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)

        c, mx = None, -1
        for i in grps.values():
            i = i[::-1]
            if c is None:
                c = i
            else:
                mx = max(mx, len(i))
                for j in range(min(len(c), len(i))):
                    if c[j] != i[j]:
                        c = i[:j]
                        break
            if len(c) == 0:
                break
        for k, i in grps.items():
            i = tuple([''] * (mx - len(i) - len(c)) +  i[:-len(c)] + [k])
            result[k] = result[k].rename(i)
        return pd.DataFrame(result.values())

    def calc(self, nodes):
        result = [self.calc_idx(nodes, i) for i in range(self.e.get_n_splits())]
        return pd.concat(result, axis=1)

In [60]:
for (y_true_train_t, y_true_valid_t), y_true_valid in e2.get_node_output(0, 'cb1'):
    print(y_true_train_t)
    print(y_true_valid_t)

In [61]:
e.nodes['cb1'].objs_[0][0][0].X_

['annual_income',
 'debt_to_income_ratio',
 'credit_score',
 'loan_amount',
 'interest_rate',
 'grade_subgrade_no',
 'gender',
 'marital_status',
 'education_level',
 'employment_status',
 'loan_purpose']

In [62]:
m = Metric(e, (None, [y]), slice(-1, None), roc_auc_score, True)
result = m.calc(['lr1', 'lr2', 'cb1'])
result

0                             1                             2  \
           train               valid     train               valid     train   
         train_0   valid_0             train_0   valid_0             train_0   
lr lr1  0.781213  0.764964  0.777506  0.771995  0.802939  0.787993  0.785705   
   lr2  0.800964  0.779320  0.801194  0.809414  0.839576  0.770695  0.809093   
cb cb1  0.941166  0.911951  0.925365  0.938722  0.938626  0.914850  0.966129   

                            
                     valid  
         valid_0            
lr lr1  0.783333  0.768093  
   lr2  0.835663  0.772863  
cb cb1  0.914267  0.907851

In [63]:
result = m.calc(['lr1', 'lr2', 'cb1'])
result

0                             1                             2  \
           train               valid     train               valid     train   
         train_0   valid_0             train_0   valid_0             train_0   
lr lr1  0.781213  0.764964  0.777506  0.771995  0.802939  0.787993  0.785705   
   lr2  0.800964  0.779320  0.801194  0.809414  0.839576  0.770695  0.809093   
cb cb1  0.941166  0.911951  0.925365  0.938722  0.938626  0.914850  0.966129   

                            
                     valid  
         valid_0            
lr lr1  0.783333  0.768093  
   lr2  0.835663  0.772863  
cb cb1  0.914267  0.907851

In [64]:
e3 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = None, splitter_params = {'y': y})

🔄 Creating new Experimenter with same structure...
   ├─ Created base Experimenter with 3 fold(s)
   ├─ Cloned 6 group(s)
[std] Building: 3/3 (100%) ✓ Complete
[lr1] Building: 3/3 (100%) ✓ Complete
[ohe] Building: 3/3 (100%) ✓ Complete
[lr2] Building: 3/3 (100%) ✓ Complete
[pca] Building: 3/3 (100%) ✓ Complete
[lr3] Building: 3/3 (100%) ✓ Complete
[cb1] Building: 3/3 (100%) ✓ Complete
  Progress: 70/100 (70.0%)))

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2139: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


[lgb1] Building: 3/3 (100%) ✓ Complete
   └─ Cloned 8 node(s)
✅ Structure cloning complete!


In [68]:
e2.desc_node_vars('lr3', 0)

[(['ohe__gender_Female',
   'ohe__gender_Male',
   'ohe__gender_Other',
   'ohe__marital_status_Divorced',
   'ohe__marital_status_Married',
   'ohe__marital_status_Single',
   'ohe__marital_status_Widowed',
   "ohe__education_level_Bachelor's",
   'ohe__education_level_High School',
   "ohe__education_level_Master's",
   'ohe__education_level_Other',
   'ohe__education_level_PhD',
   'ohe__employment_status_Employed',
   'ohe__employment_status_Retired',
   'ohe__employment_status_Self-employed',
   'ohe__employment_status_Student',
   'ohe__employment_status_Unemployed',
   'ohe__loan_purpose_Business',
   'ohe__loan_purpose_Car',
   'ohe__loan_purpose_Debt consolidation',
   'ohe__loan_purpose_Education',
   'ohe__loan_purpose_Home',
   'ohe__loan_purpose_Medical',
   'ohe__loan_purpose_Other',
   'ohe__loan_purpose_Vacation',
   'pca__pca0',
   'pca__pca1',
   'pca__pca2',
   'pca__pca3',
   'pca__pca4'],
  ['lr3__loan_paid_back_0', 'lr3__loan_paid_back_1'],
  [0, 1])]

In [119]:
m = Metric(e3, (None, [y]), slice(-1, None), roc_auc_score, False)
result = m.calc(['lr1', 'lr2', 'lr3', 'cb1'])
result

0         1         2
lr lr1  0.779186  0.788025  0.768796
   lr2  0.792669  0.774043  0.775059
   lr3  0.923389  0.912857  0.908137
cb cb1  0.924001  0.913287  0.909615

In [120]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

        cb1__loan_paid_back_0
id                           
437626               0.031421
56509                0.989099
590576               0.497218
469144               0.006575
118806               0.131097
...                       ...
167513               0.079113
14614                0.990822
490101               0.158133
341032               0.011666
247234               0.106048

[1188 rows x 1 columns]
        cb1__loan_paid_back_0
id                           
437626               0.024687
56509                0.996380
590576               0.397108
469144               0.010977
118806               0.138448
...                       ...
167513               0.100434
14614                0.995543
490101               0.222619
341032               0.011216
247234               0.123429

[1188 rows x 1 columns]


In [ ]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

In [ ]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

In [64]:
e2.get_node_train_output(0, 'lr1', slice(0, -1), method = 'mean').data

,lr1__loan_paid_back_0
id,
365017,0.654976
37601,0.083008
308896,0.672309
492362,0.211294
269337,0.090463
...,...
272044,0.105159
493452,0.319445
496963,0.018245


In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123))

In [ ]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
e3.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [ ]:
e3.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e3.set_node('lr1', 'lr')

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [ ]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [ ]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

In [ ]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e.set_node('lr1', 'lr')

In [ ]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

In [ ]:
e.get_node_output(0, 'lr1', slice(0, -1), method = 'mean').data

In [ ]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [59]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

/home/sun9sun9/python312/lib/python3.12/site-packages/sklearn/utils/validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/sun9sun9/python312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [76]:
df_train.to_pandas()[[y]].shape

(593994, 1)

In [77]:
e.nodes['lr1'].y

'loan_paid_back'